In [303]:
import pandas as pd
import numpy as np
import fancyimpute
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [269]:
df = pd.read_csv(r"E:\Imarticus_ Learning\Jobs - 2026\Models\data\unzipped\Life Expectancy Data.csv")
df.shape

(2938, 22)

In [270]:
df.columns = ['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure',
       'Diphtheria', ' HIV/AIDS', 'GDP', 'Population',
       'thinness  1-19 years', 'thinness 5-9 years',
       'Income composition of resources', 'Schooling']

In [271]:
df.columns

Index(['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure',
       'Diphtheria', ' HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
       'thinness 5-9 years', 'Income composition of resources', 'Schooling'],
      dtype='object')

In [272]:
df.value_counts('Status')

Status
Developing    2426
Developed      512
Name: count, dtype: int64

In [273]:
df = df.drop(columns = ['Country'])
encode = {
    'Developing' : 0,
    'Developed' : 1
}
df['Status'] = df['Status'].map(encode)

In [274]:
imp = fancyimpute.IterativeImputer().fit(df)

In [275]:
df_new = imp.transform(df)

In [276]:
df_new = pd.DataFrame(data= df_new, columns = df.columns)

In [277]:
x = df_new.drop(columns = 'Status')
y = df_new['Status']

In [278]:
X_train, X_test, y_train , y_test = train_test_split(x, y, test_size = 0.3, random_state = 0, stratify = y)

In [279]:
sc = StandardScaler().fit(X_train)
sc_train_array = sc.transform(X_train)
sc_train = pd.DataFrame(data = sc_train_array, columns = X_train.columns)
sc_test_array = sc.transform(X_test)
sc_test = pd.DataFrame(data = sc_test_array, columns = X_test.columns)

In [280]:
logimodel = LogisticRegression()
logimodel.fit(sc_train, y_train)

LogisticRegression()

In [281]:
logimodel.intercept_

array([-6.38019083])

In [282]:
coeff = (np.e**(logimodel.coef_)).flatten().tolist()
coeff_list = [logimodel.intercept_[0]] + coeff

print(coeff_list)

[np.float64(-6.380190825586871), 0.6501245219981139, 1.4612738655193878, 0.4236338994094686, 0.9646654276836387, 3.871436620335729, 0.9442364195657258, 1.9217767517074091, 1.086971320327328, 0.8984970114242172, 1.0155916313636153, 0.9723838813436739, 1.229287376310529, 1.0115939990416514, 0.6960872380022091, 1.1550479851144189, 0.5841932051940544, 0.29716609496123286, 1.1779237856263791, 32.80644300628594, 0.9667784195099866]


In [283]:
X_train.columns

Index(['Year', 'Life expectancy', 'Adult Mortality', 'infant deaths',
       'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI',
       'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria',
       ' HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
       'thinness 5-9 years', 'Income composition of resources', 'Schooling'],
      dtype='object')

In [284]:
coeff_df = pd.DataFrame( data = coeff_list, columns = ['Estimates'], index = ['Intercept','Year', 'Life expectancy', 'Adult Mortality', 'infant deaths',
       'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI',
       'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria',
       ' HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
       'thinness 5-9 years', 'Income composition of resources', 'Schooling'])
    

In [295]:
coeff_df.sort_values(by = 'Estimates')

,Estimates
Intercept,-6.380
thinness 1-19 years,0.297
Adult Mortality,0.424
Population,0.584
Year,0.650
HIV/AIDS,0.696
BMI,0.898
percentage expenditure,0.944
infant deaths,0.965
Schooling,0.967


In [286]:
pred_train = logimodel.predict_proba(sc_train)
pred_test = logimodel.predict_proba(sc_test)

In [287]:
pred_train.shape

(2056, 2)

In [288]:
out_class_train = []
for i in pred_train:
    max = i.max()
    pos = list(i).index(max)
    out_class_train.append(pos)

out_class_test = []
for i in pred_test:
    max = i.max()
    pos = list(i).index(max)
    out_class_test.append(pos)

In [289]:
conf_train = confusion_matrix(y_train, out_class_train)
conf_train

array([[1640,   58],
       [  61,  297]])

In [290]:
acc_train = conf_train.diagonal().sum()/conf_train.sum()
acc_train

np.float64(0.9421206225680934)

In [291]:
confusion_matrix(y_test, out_class_test)

array([[704,  24],
       [ 33, 121]])

In [292]:
print(classification_report(y_train, out_class_train))

              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96      1698
         1.0       0.84      0.83      0.83       358

    accuracy                           0.94      2056
   macro avg       0.90      0.90      0.90      2056
weighted avg       0.94      0.94      0.94      2056



In [293]:
print(classification_report(y_test, out_class_test))

              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96       728
         1.0       0.83      0.79      0.81       154

    accuracy                           0.94       882
   macro avg       0.89      0.88      0.89       882
weighted avg       0.93      0.94      0.93       882



In [314]:
solver = ['lbfgs', 'liblinear', 'newton-cg' , 'newton-cholesky', 'sag', 'saga']
for i in solver:
    lmodel = LogisticRegression(solver = i, n_jobs=-1, random_state=0, max_iter = 1000).fit(sc_train, y_train)
    pred_train = lmodel.predict(sc_train)
    pred_test = lmodel.predict(sc_test)
    acc_train = classification_report(y_train , pred_train)
    acc_test = classification_report(y_test, pred_test)
    print(f'classification_report of Train is \n{acc_train},\n classification_report of Test is \n{acc_test},\n when solver is {i}')

classification_report of Train is 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96      1698
         1.0       0.84      0.83      0.83       358

    accuracy                           0.94      2056
   macro avg       0.90      0.90      0.90      2056
weighted avg       0.94      0.94      0.94      2056
,
 classification_report of Test is 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96       728
         1.0       0.83      0.79      0.81       154

    accuracy                           0.94       882
   macro avg       0.89      0.88      0.89       882
weighted avg       0.93      0.94      0.93       882
,
 when solver is lbfgs
classification_report of Train is 
              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97      1698
         1.0       0.84      0.84      0.84       358

    accuracy                           0.94      2056

E:\anaconda3\envs\ml_3.12\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(


classification_report of Train is 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96      1698
         1.0       0.84      0.83      0.83       358

    accuracy                           0.94      2056
   macro avg       0.90      0.90      0.90      2056
weighted avg       0.94      0.94      0.94      2056
,
 classification_report of Test is 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96       728
         1.0       0.84      0.79      0.82       154

    accuracy                           0.94       882
   macro avg       0.90      0.88      0.89       882
weighted avg       0.94      0.94      0.94       882
,
 when solver is newton-cg
classification_report of Train is 
              precision    recall  f1-score   support

         0.0       0.96      0.97      0.96      1698
         1.0       0.84      0.83      0.83       358

    accuracy                           0.94      

In [311]:
from pathlib import Path
import joblib

save_dir = Path(r'E:\Imarticus_ Learning\Jobs - 2026\Models\models')
joblib.dump(logimodel, save_dir / 'Logistic_regression_model.pkl')

['E:\\Imarticus_ Learning\\Jobs - 2026\\Models\\models\\Logistic_regression_model.pkl']